# GenAI Hackathon Demo v2  
**Image → LLaVA → Reasoned Text → Auto-Summary → Bark → Expressive Audio**

This notebook is a stronger second version with:
- a cleaner Gradio UI theme
- automatic summarization before speech synthesis
- optional Hindi/Gujarati fallback text path for TTS
- downloadable architecture figure
- memory-aware defaults for hackathon demos

## Notes
- Default VLM: `llava-hf/llava-1.5-7b-hf`
- Default TTS: `suno/bark-small`
- `bitsandbytes` 4-bit loading is used for LLaVA when CUDA is available.
- Bark does **not** natively guarantee true Gujarati/Hindi pronunciation quality. This notebook includes a practical fallback path:
  - original reasoned English text
  - optional translated Hindi/Gujarati text for display
  - optional TTS source selection
- For a hackathon, English audio is usually the most reliable demo path with Bark.



In [1]:
# ============================================================
# 1. INSTALLS
# ============================================================
# Uncomment when running in a fresh environment
# !pip -q install --upgrade pip
# !pip -q install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
# !pip -q install transformers accelerate bitsandbytes sentencepiece gradio pillow matplotlib scipy
# !pip -q install langdetect deep-translator

print("Install cell ready. Uncomment if needed.")


Install cell ready. Uncomment if needed.


In [2]:
# ============================================================
# 2. IMPORTS
# ============================================================
import os
import gc
import textwrap
import warnings
from io import BytesIO

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
from transformers import (
    AutoProcessor,
    LlavaForConditionalGeneration,
    AutoTokenizer,
    BarkModel,
)
from scipy.io.wavfile import write as wav_write

import gradio as gr

# Optional utilities
try:
    from deep_translator import GoogleTranslator
    HAS_TRANSLATOR = True
except Exception:
    HAS_TRANSLATOR = False

warnings.filterwarnings("ignore")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32

print("DEVICE:", DEVICE)
print("DTYPE :", DTYPE)


DEVICE: cuda
DTYPE : torch.float16


In [3]:
# ============================================================
# 3. CONFIG
# ============================================================
LLAVA_MODEL_ID = "llava-hf/llava-1.5-7b-hf"
BARK_MODEL_ID = "suno/bark-small"

DEFAULT_SYSTEM_STYLE = (
    "You are a helpful multimodal assistant. "
    "Analyze the image carefully, reason step by step internally, "
    "and provide a concise but useful answer for a live demo."
)

VOICE_PRESETS = [
    "v2/en_speaker_0",
    "v2/en_speaker_1",
    "v2/en_speaker_3",
    "v2/en_speaker_6",
    "v2/en_speaker_9",
]

LANGUAGE_OPTIONS = ["English", "Hindi", "Gujarati"]
TTS_SOURCE_OPTIONS = [
    "Auto Summary (recommended)",
    "Full Reasoned Text",
    "Translated Summary",
]


In [4]:
# ============================================================
# 4. MODEL LOADERS
# ============================================================
llava_processor = None
llava_model = None
bark_processor = None
bark_model = None

def load_llava():
    global llava_processor, llava_model
    if llava_model is not None:
        return llava_processor, llava_model

    llava_processor = AutoProcessor.from_pretrained(LLAVA_MODEL_ID)

    if DEVICE == "cuda":
        llava_model = LlavaForConditionalGeneration.from_pretrained(
            LLAVA_MODEL_ID,
            torch_dtype=torch.float16,
            low_cpu_mem_usage=True,
            load_in_4bit=True,
            device_map="auto",
        )
    else:
        llava_model = LlavaForConditionalGeneration.from_pretrained(
            LLAVA_MODEL_ID,
            torch_dtype=torch.float32,
            low_cpu_mem_usage=True,
        ).to(DEVICE)

    return llava_processor, llava_model


def load_bark():
    global bark_processor, bark_model
    if bark_model is not None:
        return bark_processor, bark_model

    bark_processor = AutoProcessor.from_pretrained(BARK_MODEL_ID)
    bark_model = BarkModel.from_pretrained(BARK_MODEL_ID)
    bark_model = bark_model.to(DEVICE)

    return bark_processor, bark_model


In [5]:
# ============================================================
# 5. PROMPTING HELPERS
# ============================================================
def build_llava_prompt(user_prompt: str, style: str = DEFAULT_SYSTEM_STYLE):
    user_prompt = user_prompt.strip()
    if not user_prompt:
        user_prompt = (
            "Describe the image, identify important objects or text, "
            "and explain the scene in a clear and useful way."
        )

    prompt = (
        f"USER: <image>\n"
        f"System style: {style}\n"
        f"Task: {user_prompt}\n"
        f"ASSISTANT:"
    )
    return prompt


def make_summary_text(reasoned_text: str, max_sentences: int = 2, max_chars: int = 280):
    '''
    Lightweight hackathon-friendly summary without an extra LLM call.
    Keeps latency low. Extractive compression by sentence splitting.
    '''
    text = reasoned_text.strip().replace("\n", " ")
    parts = [p.strip() for p in text.replace("!", ".").replace("?", ".").split(".") if p.strip()]

    if not parts:
        return text[:max_chars]

    summary = ". ".join(parts[:max_sentences]).strip()
    if not summary.endswith("."):
        summary += "."
    if len(summary) > max_chars:
        summary = summary[:max_chars].rsplit(" ", 1)[0] + "..."
    return summary


In [6]:
# ============================================================
# 6. TRANSLATION HELPERS
# ============================================================
def translate_text(text: str, target_language: str):
    '''
    Practical display fallback for Hindi/Gujarati.
    Requires internet in runtime for GoogleTranslator path.
    If unavailable, returns the original text with a note.
    '''
    if target_language == "English":
        return text

    language_map = {
        "Hindi": "hi",
        "Gujarati": "gu",
    }

    if not HAS_TRANSLATOR:
        return f"[Translation unavailable in current environment] {text}"

    try:
        return GoogleTranslator(source="auto", target=language_map[target_language]).translate(text)
    except Exception as e:
        return f"[Translation failed: {e}] {text}"


In [7]:
# ============================================================
# 7. CORE INFERENCE: IMAGE -> LLaVA
# ============================================================
@torch.inference_mode()
def llava_reason(image: Image.Image, user_prompt: str, max_new_tokens: int = 180):
    processor, model = load_llava()

    image = image.convert("RGB")
    prompt = build_llava_prompt(user_prompt)

    inputs = processor(text=prompt, images=image, return_tensors="pt")

    for k, v in inputs.items():
        if torch.is_tensor(v):
            inputs[k] = v.to(model.device)

    generated = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.3,
        top_p=0.9,
        repetition_penalty=1.1,
    )

    text = processor.batch_decode(generated, skip_special_tokens=True)[0]

    # Post-cleaning: remove echoed prompt if present
    if "ASSISTANT:" in text:
        text = text.split("ASSISTANT:", 1)[-1].strip()

    return text


In [8]:
# ============================================================
# 8. CORE INFERENCE: TEXT -> BARK AUDIO
# ============================================================
@torch.inference_mode()
def bark_tts(text: str, voice_preset: str = "v2/en_speaker_6", output_wav="bark_output.wav"):
    processor, model = load_bark()

    text = text.strip()
    if not text:
        raise ValueError("Empty text cannot be converted to speech.")

    inputs = processor(
        text,
        voice_preset=voice_preset,
        return_tensors="pt",
    )

    for k, v in inputs.items():
        if torch.is_tensor(v):
            inputs[k] = v.to(model.device)

    audio_array = model.generate(**inputs)
    audio_array = audio_array.cpu().numpy().squeeze()

    sample_rate = model.generation_config.sample_rate
    wav_write(output_wav, rate=sample_rate, data=(audio_array * 32767).astype(np.int16))

    return output_wav, sample_rate


In [9]:
# ============================================================
# 9. ARCHITECTURE FIGURE GENERATOR
# ============================================================
def save_architecture_figure(path="genai_hackathon_architecture_v2.png"):
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.axis("off")

    boxes = [
        (0.03, 0.35, 0.14, 0.3, "Input Image"),
        (0.22, 0.35, 0.18, 0.3, "LLaVA\nVision-Language\nReasoning"),
        (0.45, 0.35, 0.16, 0.3, "Reasoned Text"),
        (0.66, 0.35, 0.15, 0.3, "Auto Summary"),
        (0.85, 0.35, 0.12, 0.3, "Bark\nExpressive Audio"),
    ]

    for x, y, w, h, label in boxes:
        rect = plt.Rectangle((x, y), w, h, fill=False, linewidth=2)
        ax.add_patch(rect)
        ax.text(x + w/2, y + h/2, label, ha="center", va="center", fontsize=12)

    arrows = [
        (0.17, 0.5, 0.22, 0.5),
        (0.40, 0.5, 0.45, 0.5),
        (0.61, 0.5, 0.66, 0.5),
        (0.81, 0.5, 0.85, 0.5),
    ]
    for x1, y1, x2, y2 in arrows:
        ax.annotate("", xy=(x2, y2), xytext=(x1, y1),
                    arrowprops=dict(arrowstyle="->", linewidth=2))

    ax.text(0.5, 0.88, "GenAI Hackathon Pipeline v2", ha="center", fontsize=16, fontweight="bold")
    ax.text(0.5, 0.15, "Optional display fallback: translated Hindi/Gujarati summary for UI output", ha="center", fontsize=11)

    plt.tight_layout()
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    return path

arch_path = save_architecture_figure()
print("Saved:", arch_path)


Saved: genai_hackathon_architecture_v2.png


In [10]:
# ============================================================
# 10. END-TO-END PIPELINE
# ============================================================
def run_pipeline(
    image,
    prompt,
    voice_preset,
    target_language,
    tts_source_mode,
    max_new_tokens=180,
):
    if image is None:
        raise gr.Error("Please upload an image.")

    if isinstance(image, np.ndarray):
        image = Image.fromarray(image)

    reasoned_text = llava_reason(image, prompt, max_new_tokens=max_new_tokens)
    summary_text = make_summary_text(reasoned_text)
    translated_summary = translate_text(summary_text, target_language)

    if tts_source_mode == "Auto Summary (recommended)":
        tts_text = summary_text
    elif tts_source_mode == "Full Reasoned Text":
        tts_text = reasoned_text
    else:
        tts_text = translated_summary

    # Practical Bark stability guardrail
    if target_language in ["Hindi", "Gujarati"] and tts_source_mode == "Translated Summary":
        # Bark may not pronounce these reliably. Still allowed for demo experimentation.
        pass

    wav_path, sr = bark_tts(tts_text, voice_preset=voice_preset, output_wav="demo_output.wav")

    return reasoned_text, summary_text, translated_summary, wav_path, arch_path


In [11]:
# ============================================================
# 11. GRADIO APP
# ============================================================
CUSTOM_CSS = '''
.gradio-container {
    max-width: 1400px !important;
}
h1, h2, h3 {
    letter-spacing: 0.2px;
}
.footer-note {
    font-size: 0.9rem;
    opacity: 0.8;
}
'''

with gr.Blocks(theme=gr.themes.Soft(), css=CUSTOM_CSS, title="GenAI Vision-to-Voice Demo v2") as demo:
    gr.Markdown(
        """
# GenAI Hackathon Demo v2
**Image → LLaVA → Reasoned Text → Auto Summary → Bark → Expressive Audio**

Upload an image, ask a question or instruction, and generate:
1. reasoned multimodal output,
2. compact summary,
3. optional Hindi/Gujarati translated summary,
4. expressive audio,
5. downloadable architecture figure.
"""
    )

    with gr.Row():
        with gr.Column(scale=1):
            image_input = gr.Image(type="pil", label="Upload Image")
            prompt_input = gr.Textbox(
                label="Prompt / Task",
                lines=4,
                value="Explain what is happening in this image. Mention important objects, text if visible, and the likely context."
            )
            voice_input = gr.Dropdown(VOICE_PRESETS, value="v2/en_speaker_6", label="Bark Voice Preset")
            lang_input = gr.Dropdown(LANGUAGE_OPTIONS, value="English", label="Display Language")
            tts_mode_input = gr.Dropdown(TTS_SOURCE_OPTIONS, value="Auto Summary (recommended)", label="Speech Source")
            max_tokens_input = gr.Slider(minimum=64, maximum=256, step=16, value=160, label="Max New Tokens")
            run_btn = gr.Button("Run Demo", variant="primary")

        with gr.Column(scale=1):
            reasoned_output = gr.Textbox(label="Reasoned Text", lines=10)
            summary_output = gr.Textbox(label="Auto Summary", lines=4)
            translated_output = gr.Textbox(label="Translated Summary", lines=4)
            audio_output = gr.Audio(label="Generated Speech", type="filepath")
            arch_output = gr.File(label="Download Architecture Figure")

    run_btn.click(
        fn=run_pipeline,
        inputs=[image_input, prompt_input, voice_input, lang_input, tts_mode_input, max_tokens_input],
        outputs=[reasoned_output, summary_output, translated_output, audio_output, arch_output],
    )

    gr.Markdown(
        """
<div class="footer-note">
Recommended live demo mode: English + Auto Summary.
Hindi/Gujarati translation is best treated as UI/display support unless you validate TTS quality in your runtime.
</div>
"""
    )

demo


Gradio Blocks instance: 1 backend functions
-------------------------------------------
fn_index=0
 inputs:
 |-<gradio.components.image.Image object at 0x7d3ded6d1970>
 |-<gradio.components.textbox.Textbox object at 0x7d3dea2606b0>
 |-<gradio.components.dropdown.Dropdown object at 0x7d3ded679eb0>
 |-<gradio.components.dropdown.Dropdown object at 0x7d3dea298800>
 |-<gradio.components.dropdown.Dropdown object at 0x7d3dea29ab70>
 |-<gradio.components.slider.Slider object at 0x7d3dea24ba40>
 outputs:
 |-<gradio.components.textbox.Textbox object at 0x7d3ded9bf320>
 |-<gradio.components.textbox.Textbox object at 0x7d3ded673c80>
 |-<gradio.components.textbox.Textbox object at 0x7d3dea2c9130>
 |-<gradio.components.audio.Audio object at 0x7d3dea23dca0>
 |-<gradio.components.file.File object at 0x7d3dea24b4d0>

In [12]:
# ============================================================
# 12. LAUNCH APP
# ============================================================
# For local/Jupyter use:
# demo.launch()

# For public temporary sharing in a hackathon setting:
# demo.launch(share=True)

print("Run demo.launch() when ready.")


Run demo.launch() when ready.


In [13]:
# ============================================================
# 13. OPTIONAL QUICK TEST WITHOUT GRADIO
# ============================================================
# Example:
# sample = Image.open("your_image.jpg").convert("RGB")
# reasoned, summary, translated, wav_path, fig_path = run_pipeline(
#     image=sample,
#     prompt="Describe the image and explain the important details.",
#     voice_preset="v2/en_speaker_6",
#     target_language="English",
#     tts_source_mode="Auto Summary (recommended)",
#     max_new_tokens=160
# )
# print("Reasoned:", reasoned)
# print("Summary :", summary)
# print("Translated:", translated)
# print("Audio:", wav_path)
# print("Figure:", fig_path)


## Practical Hackathon Notes

### Best demo settings
- **Language:** English
- **Speech source:** Auto Summary
- **Voice preset:** `v2/en_speaker_6`
- **Max new tokens:** 128–160

### Why this version is stronger
- Reduced Bark input length improves audio latency and stability.
- Summary-first TTS is more presentation-friendly.
- Architecture figure is generated automatically for your PPT/demo.
- Translation field helps multilingual storytelling even when TTS remains English-first.

### Upgrade path after this
- Replace summary heuristic with an LLM summarizer.
- Add OCR pre-pass for text-heavy documents.
- Add response caching.
- Add downloadable PDF report of image + explanation + summary.
